# Capitolo 7: Creazione di Applicazioni Chat
## Guida rapida all'API OpenAI

Questo notebook è adattato dal [Repository di Esempi Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) che include notebook che accedono ai servizi [Azure OpenAI](notebook-azure-openai.ipynb).

L'API Python di OpenAI funziona anche con i Modelli Azure OpenAI, con alcune modifiche. Scopri di più sulle differenze qui: [Come passare tra endpoint OpenAI e Azure OpenAI con Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Panoramica  
"I grandi modelli linguistici sono funzioni che mappano testo su testo. Dato un input testuale, un grande modello linguistico cerca di prevedere il testo che verrà dopo"(1). Questo notebook "quickstart" introdurrà gli utenti ai concetti di alto livello degli LLM, ai requisiti di base del pacchetto per iniziare con AML, a una introduzione leggera al design dei prompt e a diversi brevi esempi di vari casi d'uso. 


## Indice  

[Panoramica](#overview)  
[Come utilizzare il servizio OpenAI](#how-to-use-openai-service)  
[1. Creazione del tuo servizio OpenAI](#1.-creating-your-openai-service)  
[2. Installazione](#2.-installation)    
[3. Credenziali](#3.-credentials)  

[Casi d'uso](#use-cases)    
[1. Riassumere testo](#1.-summarize-text)  
[2. Classificare testo](#2.-classify-text)  
[3. Generare nuovi nomi di prodotto](#3.-generate-new-product-names)  
[4. Affinare un classificatore](#4.fine-tune-a-classifier)  

[Riferimenti](#references)


### Crea il tuo primo prompt  
Questo breve esercizio fornirà una introduzione di base per l'invio di prompt a un modello OpenAI per un compito semplice di "sintesi".  


**Passaggi**:  
1. Installa la libreria OpenAI nel tuo ambiente python  
2. Carica le librerie di supporto standard e imposta le tue credenziali di sicurezza tipiche di OpenAI per il servizio OpenAI che hai creato  
3. Scegli un modello per il tuo compito  
4. Crea un prompt semplice per il modello  
5. Invia la tua richiesta all'API del modello!  


### 1. Installa OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importa librerie di supporto e crea le credenziali


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Trovare il modello giusto  
Modelli come GPT-4o e GPT-4o mini possono comprendere e generare linguaggio naturale, e sono disponibili sulla piattaforma OpenAI con diversi livelli di potenza e velocità adatti a compiti differenti.


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. Progettazione del Prompt  

"La magia dei grandi modelli di linguaggio è che, essendo addestrati per minimizzare questo errore di previsione su enormi quantità di testo, i modelli finiscono per apprendere concetti utili per queste previsioni. Per esempio, imparano concetti come"(1):

* come si scrive correttamente
* come funziona la grammatica
* come parafrasare
* come rispondere a domande
* come sostenere una conversazione
* come scrivere in molte lingue
* come programmare
* ecc.

#### Come controllare un grande modello di linguaggio  
"Di tutti gli input per un grande modello di linguaggio, di gran lunga il più influente è il prompt di testo(1).

I grandi modelli di linguaggio possono essere stimolati a produrre output in alcuni modi:

Istruzione: Dì al modello cosa vuoi
Completamento: Induci il modello a completare l’inizio di ciò che vuoi
Dimostrazione: Mostra al modello cosa vuoi, con uno dei seguenti:
Alcuni esempi nel prompt
Centinaia o migliaia di esempi in un dataset di addestramento per il fine-tuning"



#### Ci sono tre linee guida di base per creare prompt:

**Mostra e dì**. Rendilo chiaro cosa vuoi, sia attraverso istruzioni, esempi o una combinazione dei due. Se vuoi che il modello ordini alfabeticamente una lista di elementi o classifichi un paragrafo per sentimento, mostrargli che è quello che vuoi.

**Fornisci dati di qualità**. Se stai cercando di costruire un classificatore o di far seguire al modello un modello preciso, assicurati che ci siano abbastanza esempi. Assicurati di rileggere i tuoi esempi — il modello è di solito abbastanza intelligente da vedere attraverso errori ortografici di base e darti una risposta, ma può anche presumere che sia intenzionale e questo può influenzare la risposta.

**Controlla le tue impostazioni.** Le impostazioni temperature e top_p controllano quanto il modello è deterministico nel generare una risposta. Se stai chiedendo una risposta in cui c’è una sola risposta giusta, allora vorresti impostare questi valori più bassi. Se vuoi risposte più varie, allora potrebbe volerli impostare più alti. L’errore principale che le persone fanno con queste impostazioni è presumere che siano controlli di "intelligenza" o "creatività".


Fonte: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Invia!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ripetere la stessa chiamata, come si confrontano i risultati?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Riassumere il Testo  
#### Sfida  
Riassumi il testo aggiungendo un 'tl;dr:' alla fine di un passaggio di testo. Nota come il modello comprende come eseguire numerosi compiti senza istruzioni aggiuntive. Puoi sperimentare con prompt più descrittivi di tl;dr per modificare il comportamento del modello e personalizzare il riassunto che ricevi(3).  

Lavori recenti hanno dimostrato notevoli miglioramenti in molti compiti e benchmark di NLP pre-addestrando su un ampio corpus di testi seguito da un fine-tuning su un compito specifico. Sebbene tipicamente agnostico al compito nell'architettura, questo metodo richiede ancora dataset di fine-tuning specifici per il compito di migliaia o decine di migliaia di esempi. Al contrario, gli esseri umani in genere possono eseguire un nuovo compito linguistico con solo pochi esempi o con istruzioni semplici - qualcosa con cui i sistemi NLP attuali fanno ancora molta fatica. Qui mostriamo che aumentare la scala dei modelli linguistici migliora notevolmente le prestazioni agnostiche al compito e few-shot, a volte raggiungendo anche una competitività con i precedenti approcci di fine-tuning allo stato dell'arte.  



Tl;dr


# Esercizi per diversi casi d'uso  
1. Riassumere testo  
2. Classificare testo  
3. Generare nuovi nomi di prodotti


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Classificare Testo  
#### Sfida  
Classificare gli elementi nelle categorie fornite al momento dell'inferenza. Nell'esempio seguente, forniamo sia le categorie sia il testo da classificare nel prompt (*playground_reference). 

Richiesta del cliente: Ciao, una delle tastiere del mio laptop si è rotta di recente e avrò bisogno di una sostituzione:

Categoria classificata:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Genera Nuovi Nomi per Prodotti
#### Sfida
Crea nomi per prodotti a partire da parole di esempio. Qui includiamo nel prompt informazioni sul prodotto per cui vogliamo generare nomi. Forniamo inoltre un esempio simile per mostrare lo schema che desideriamo ricevere. Abbiamo anche impostato un valore elevato per la temperatura per aumentare la casualità e ottenere risposte più innovative.

Descrizione del prodotto: una macchina per milkshake domestica
Parole seed: veloce, sano, compatto.
Nomi prodotto: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descrizione del prodotto: un paio di scarpe che si adattano a qualsiasi misura di piede.
Parole seed: adattabile, adatto, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Riferimenti  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Per Ulteriore Aiuto  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Collaboratori
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
